# 随机游走 Metropolis-Hastings

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/MCMC_random_walk.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用随机游走提议分布从目标分布中抽样，去掉前20%的预热样本后计算两个积分。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**随机游走的Metropolis-Hastings算法示例**

清空、关闭分页、固定种子，并设抽样次数 M=10000。


In [ ]:
%%stata
clear all
set more off
set seed 8855


**设定参数**

本段实现**随机游走的 Metropolis-Hastings 算法**，并用它计算两个积分。结构：

- `pai(x)=exp(-90(x1-0.5)²-45(x2+0.1)²)`：目标密度（**未归一化**——MH 算法只需要密度的比例，不需要归一化常数，这是它相对拒绝抽样的重要优势）；
- `h(x)`、`h2(x)`：两个待积分的函数（定义域指示函数；sin²+log 形式的函数）；
- `sample_q(x)=x+0.2·rnormal(1,2,0,1)`：**随机游走提议分布**——在当前位置 x 附近加一个小扰动（每维标准差0.2），因此提议点与当前点相关，这是与独立 MH 的关键区别；
- `MH_RW(...)`：MH 算法主体——接受概率 ρ=min{1, π(y)/π(x)}（对称提议下 q 项抵消）；被拒绝时链停留在原地（代码里不更新 x，但仍在 X 中记录当前状态）；
- 执行抽样后，对每个样本点计算两个函数值 H、H2 并**丢弃前20%的样本**（burn-in/预热期，`mean(H[(burn+1)..M])`）——链从初始点 (0,0) 出发需要一段时间才收敛到目标分布，预热期样本不代表目标分布；
- 两点乘积 π/(90√2) 是目标密度未归一化部分对应的常数（与第一节的积分常数相同）；`printf` 输出两个积分的估计值。

顺便体会：MCMC 的样本之间存在相关性（与简单 Monte Carlo 不同），因此有效样本量小于 M。


In [ ]:
%%stata
local M = 10000

mata:
// 目标密度（未归一化）
real scalar pai(real rowvector x)
{
	return(exp(-90*(x[1]-0.5)^2 - 45*(x[2]+0.1)^2))
}

// 被积函数：定义域指示函数
real scalar h(real rowvector x)
{
	return(x[1]>=-1 && x[1]<=1 && x[2]>=-1 && x[2]<=1)
}

// 另一个被积函数
real scalar h2(real rowvector x)
{
	return(sin(10*x[1])^2 + log(abs(1+x[2]*10)))
}

// 随机游走的提议分布：给定x，从q(y|x)中抽样
real rowvector sample_q(real rowvector x)
{
	return(x + 0.2*rnormal(1, 2, 0, 1))
}

// 随机游走的MCMC算法，输入：
//    N_samples   : 抽样次数
//      pai(x)    : 目标密度函数
//   q_sampler(x) : 给定x，从q中抽样的函数
//      x0        : 初始值
real matrix MH_RW(real scalar N_samples, pointer pai, pointer q_sampler,
                  real rowvector x0)
{
	real matrix X
	real rowvector x, y
	real scalar i, rho, u
	X = J(N_samples, length(x0), .)
	x = x0
	for (i=1; i<=N_samples; i++) {
		y = (*q_sampler)(x)
		rho = min((1, (*pai)(y)/(*pai)(x)))
		u = runiform(1, 1)
		if (u < rho) {
			x = y
		}
		X[i, .] = x
	}
	return(X)
}

// 使用随机游走的Metropolis-Hastings算法从pai(x)中抽样
X = MH_RW(`M', &pai(), &sample_q(), (0,0))

// 计算两个被积函数在每条链上的取值
H = J(`M', 1, .)
H2 = J(`M', 1, .)
for (i=1; i<=`M'; i++) {
	H[i] = h(X[i, .])
	H2[i] = h2(X[i, .])
}

// 计算积分：去掉前20%的burn-in，取后面80%的样本
burn = floor(`M'*0.2)
// 第一个积分
integral = pi()/(90*sqrt(2))*mean(H[(burn+1)..`M'])
printf("Integral1 = %f\n", integral)
// 第二个积分
integral2 = pi()/(90*sqrt(2))*mean(H2[(burn+1)..`M'])
printf("Integral2 = %f\n", integral2)
end
